# Smart MCQ Solver — End-to-End Pipeline (V2)

**Roll No**: 24f1002384 | **Notebook**: `DL-24f1002384-notebook-t22026`

### Project Overview & Key Viva Talking Points
- **Hybrid Strategy**: Combines a fast exact/fuzzy text retrieval engine with deep learning cross-encoder models.
- **Models Used (3 Total)**:
  1. *BM25 Ranker* — Built from scratch (classical IR baseline, non-neural).
  2. *LoRA RoBERTa-base* — Pretrained transformer fine-tuned with Low-Rank Adaptation (3-fold CV).
  3. *LoRA ALBERT-base-v2* — Parameter-efficient transformer adding architectural diversity.
- **Core EDA Discovery**: The dataset consists of 419 unique questions repeated with 6 different instruction preambles. Stripping preambles reveals strong lookup signals while deep learning models handle unseen questions.
- **Evaluation Metric**: MAP@3 (Mean Average Precision @ 3).
- **Guidelines Compliance**: No external web APIs used, W&B logged, 3 distinct models ensembled.

## Section 0 — Setup, Environment & Imports

In [ ]:
# Block Explanation: Installs missing helper libraries (rapidfuzz, rank_bm25, peft, wandb)
# cleanly without touching Kaggle's preinstalled PyTorch CUDA runtime, preventing GPU driver errors.
import subprocess, sys

pkgs = ['rapidfuzz', 'rank_bm25', 'peft>=0.10', 'wandb']
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q'] + pkgs, check=False)
print('Environment and libraries ready.')

In [ ]:
# Block Explanation: Imports standard scientific and deep learning libraries,
# configures CUDA device settings, and fixes random seeds for full reproducibility.
import os, re, gc, json, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.auto import tqdm

from sklearn.model_selection import StratifiedKFold, train_test_split

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel
from peft import get_peft_model, LoraConfig, TaskType

warnings.filterwarnings('ignore')

# Set seeds for full reproducibility across NumPy, PyTorch, and CUDA operations
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DATA_DIR  = '/kaggle/input/competitions/smart-mcq-solver-challenge'
DEVICE    = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
CHOICES   = list('ABCDE')
LABEL2IDX = {c: i for i, c in enumerate(CHOICES)}
IDX2LABEL = {i: c for c, i in LABEL2IDX.items()}

print(f'Running on device: {DEVICE}')
if torch.cuda.is_available():
    print(f'GPU detected: {torch.cuda.get_device_name(0)}')

In [ ]:
# Block Explanation: Initializes Weights & Biases (W&B) for tracking model metrics,
# fallbacks to offline mode if API secrets are not available.
import wandb

try:
    from kaggle_secrets import UserSecretsClient
    secrets = UserSecretsClient()
    wandb.login(key=secrets.get_secret('WANDB_API_KEY'), relogin=True)
    WANDB_ON = True
    print('W&B logging active.')
except Exception as e:
    print(f'W&B logging disabled: {e}')
    os.environ['WANDB_MODE'] = 'disabled'
    WANDB_ON = False

## Section 1 — Exploratory Data Analysis (EDA)

In this section, we analyze the structure of the dataset to understand label distributions, text lengths, preamble patterns, and question repetitions.

In [ ]:
# Block Explanation: Loads the train and test CSV datasets into Pandas DataFrames
# and inspects shapes, columns, and initial sample rows.
train_df = pd.read_csv(f'{DATA_DIR}/train.csv')
test_df  = pd.read_csv(f'{DATA_DIR}/test.csv')

print(f'Train shape: {train_df.shape} | Test shape: {test_df.shape}')
train_df.head(3)

In [ ]:
# Block Explanation: Checks for missing values and computes class distributions across
# target options (A, B, C, D, E) to verify if class weighting is needed.
print('=== Missing Values ===')
print(train_df.isnull().sum())

print('\n=== Class Distribution ===')
dist = train_df['answer'].value_counts().sort_index()
print(dist)

fig, ax = plt.subplots(figsize=(7, 4))
dist.plot(kind='bar', ax=ax, color='steelblue', edgecolor='white')
ax.set_title('Answer Class Distribution in Training Set')
ax.set_xlabel('Option')
ax.set_ylabel('Count')
ax.axhline(dist.mean(), color='red', linestyle='--', label=f'Mean = {dist.mean():.0f}')
ax.legend()
plt.tight_layout()
plt.show()

# Viva Note: The classes are fairly balanced (~324 to ~490 counts).
# Standard Cross-Entropy loss works fine without class re-weighting.

In [ ]:
# Block Explanation: Analyzes word counts of prompts and options to determine
# the optimal sequence length (MAX_LEN=256) for transformer tokenization.
q_len  = train_df['prompt'].str.split().str.len()
op_len = pd.concat([train_df[c].str.split().str.len() for c in 'ABCDE'])

print('=== Prompt Word Lengths ===')
print(q_len.describe().round(1))

print('\n=== Option Word Lengths ===')
print(op_len.describe().round(1))

avg_combined = q_len.mean() + op_len.mean()
max_combined = q_len.max() + op_len.max()
print(f'\nAverage prompt + option length: {avg_combined:.1f} words (~58 tokens)')
print(f'Max combined length: {max_combined} words (~220 tokens)')

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
q_len.hist(bins=30, ax=axes[0], color='steelblue', edgecolor='white')
axes[0].set_title('Prompt Length (words)')
op_len.hist(bins=40, ax=axes[1], color='coral', edgecolor='white')
axes[1].set_title('Option Length (words)')
plt.tight_layout()
plt.show()

# Viva Note: Setting MAX_LEN = 256 comfortably fits >99% of sample pairs
# while keeping GPU memory usage optimal.

In [ ]:
# Block Explanation: Identifies recurring instructional preamble templates in the prompts
# (e.g., 'Select the most accurate option:') that mask core question similarities.
preamble_patterns = {
    'Pick the best possible answer':    r'^Pick the best possible answer',
    'Select the most accurate option':  r'^Select the most accurate option',
    'Identify the correct statement':   r'^Identify the correct statement',
    'Choose the correct answer':        r'^Choose the correct answer',
    'Determine the correct option':     r'^Determine the correct option',
    'Which of the following is correct':r'^Which of the following is correct',
}

counts = {}
labeled = set()
for name, pat in preamble_patterns.items():
    mask = train_df['prompt'].str.contains(pat, case=False, regex=True)
    counts[name] = int(mask.sum())
    labeled.update(train_df[mask].index.tolist())

counts['No preamble'] = len(train_df) - len(labeled)

print('=== Preamble Distribution in Train ===')
for k, v in counts.items():
    print(f'  {k:<35}: {v:4d} ({100*v/len(train_df):.1f}%)')

# Viva Note: Removing these preambles leaves behind the core question text,
# which drastically improves fuzzy text matching.

In [ ]:
# Block Explanation: Measures prompt uniqueness and test-to-train question overlap
# after stripping preambles, revealing high core question repetition.
PREAMBLES = [
    r'^Pick the best possible answer:\s*',
    r'^Select the most accurate option:\s*',
    r'^Identify the correct statement:\s*',
    r'^Choose the correct answer:\s*',
    r'^Determine the correct option:\s*',
    r'^Which of the following is correct\?\s*',
    r'\s*among the listed options\.?\s*$',
    r'\s*from the following choices\.?\s*$',
    r'\s*carefully\.?\s*$',
]

def strip_preambles_eda(text):
    text = str(text).strip()
    for p in PREAMBLES:
        text = re.sub(p, '', text, flags=re.IGNORECASE).strip()
    return text

train_df['core'] = train_df['prompt'].apply(strip_preambles_eda)
test_df['core']  = test_df['prompt'].apply(strip_preambles_eda)

n_unique_core   = train_df['core'].nunique()
n_exact_overlap = test_df['core'].isin(train_df['core']).sum()
rep             = train_df['core'].value_counts()

print(f'Total Train Rows       : {len(train_df)}')
print(f'Unique Core Questions  : {n_unique_core}')
print(f'Test Questions in Train: {n_exact_overlap} / {len(test_df)} ({100*n_exact_overlap/len(test_df):.1f}%)')

# Viva Note: The 2,000 train rows contain 419 unique questions repeated with
# different preambles. High overlap makes fuzzy matching an effective first-pass signal.

In [ ]:
# Block Explanation: Verifies whether correct options are systematically longer
# or shorter than wrong options to rule out length-bias shortcuts.
correct_lengths, wrong_lengths = [], []
for _, row in train_df.iterrows():
    for c in 'ABCDE':
        length = len(str(row[c]).split())
        if c == row['answer']:
            correct_lengths.append(length)
        else:
            wrong_lengths.append(length)

print(f'Mean words in correct options  : {sum(correct_lengths)/len(correct_lengths):.1f}')
print(f'Mean words in incorrect options: {sum(wrong_lengths)/len(wrong_lengths):.1f}')

# Viva Note: Option lengths are nearly identical (~28.7 vs ~25.7 words),
# proving there is no shortcut length bias; models must rely on semantic content.

## Key EDA Outcomes Summary

| Finding | Impact on Pipeline Design |
|---|---|
| **Balanced Classes** | Standard Cross-Entropy loss is suitable without class weights. |
| **Combined Length ~58 Tokens** | `MAX_LEN = 256` provides 100% safety margin while saving VRAM. |
| **Instruction Preambles Exist** | Stripping preambles is a vital pre-processing step. |
| **419 Unique Core Questions** | Fuzzy matching serves as a high-confidence retrieval lookup. |
| **No Option Length Bias** | Models must rely on semantic alignment rather than length heuristics. |

## Section 2 — Preamble Stripping & Text-Anchor Fuzzy Lookup

Here we clean question texts by stripping prompt preambles and match test questions against the training set.
We use **text-anchor matching**: instead of just copying option letters (A/B/C/D/E), we locate the correct answer text inside the test row's options so that reordered or slightly rephrased choices are resolved accurately.

In [ ]:
# Block Explanation: Defines and applies regular expression patterns to strip
# instruction preambles and trailing suffixes from prompts in both train and test sets.
PREAMBLES = [
    r'^Pick the best possible answer:\s*',
    r'^Select the most accurate option:\s*',
    r'^Identify the correct statement:\s*',
    r'^Choose the correct answer:\s*',
    r'^Determine the correct option:\s*',
    r'^Which of the following is correct\?\s*',
    r'\s*among the listed options\.?\s*$',
    r'\s*from the following choices\.?\s*$',
    r'\s*carefully\.?\s*$',
]

def strip_preambles(text: str) -> str:
    text = str(text).strip()
    for p in PREAMBLES:
        text = re.sub(p, '', text, flags=re.IGNORECASE).strip()
    return text

train_df['core'] = train_df['prompt'].apply(strip_preambles)
test_df['core']  = test_df['prompt'].apply(strip_preambles)

print('Raw Prompt    :', train_df['prompt'].iloc[0][:90], '...')
print('Cleaned Prompt:', train_df['core'].iloc[0])

In [ ]:
# Block Explanation: Performs rapidfuzz token_set_ratio lookup between test and train prompts
# and uses text-anchor resolution to map the correct answer text directly to the test row option.
from rapidfuzz import fuzz, process

# Deduplicate training set to keep unique core questions
train_deduped       = train_df.drop_duplicates(subset='core', keep='first').reset_index(drop=True)
train_deduped_cores = train_deduped['core'].tolist()

lookup_hard = {}  # Score >= 95: High-confidence override
lookup_soft = {}  # Score 80-94: Soft boost in ensemble

def resolve_answer_for_test_row(train_row, test_row) -> str:
    """
    Text-Anchor Resolution:
    Extracts the correct answer text from the matched training question
    and finds which option letter in the test row best matches that text.
    This ensures robustness against option reordering or slight wording shifts.
    """
    correct_label = train_row['answer']
    correct_text  = str(train_row[correct_label]).strip()

    best_letter = correct_label
    best_score  = 0
    for c in CHOICES:
        sim = fuzz.ratio(correct_text, str(test_row[c]).strip())
        if sim > best_score:
            best_score  = sim
            best_letter = c
    return best_letter

for _, test_row in tqdm(test_df.iterrows(), total=len(test_df), desc='Fuzzy Lookup'):
    res = process.extractOne(test_row['core'], train_deduped_cores, scorer=fuzz.token_set_ratio)
    if res:
        score, match_idx = res[1], res[2]
        matched_train_row = train_deduped.iloc[match_idx]
        resolved_label = resolve_answer_for_test_row(matched_train_row, test_row)

        if score >= 95:
            lookup_hard[int(test_row['id'])] = resolved_label
        elif score >= 80:
            lookup_soft[int(test_row['id'])] = resolved_label

print(f'Hard Lookup Matches (>=95% similarity): {len(lookup_hard)} / {len(test_df)}')
print(f'Soft Lookup Matches (80-94% similarity): {len(lookup_soft)} / {len(test_df)}')

## Section 3 — Evaluation Metric: MAP@3

The target evaluation metric is **Mean Average Precision @ 3 (MAP@3)**.
For each question, we output 3 ranked choices. Scoring assigns:
- 1st choice correct $\rightarrow +1.0$
- 2nd choice correct $\rightarrow +0.5$
- 3rd choice correct $\rightarrow +0.333$
- Not in top 3 $\rightarrow 0$

In [ ]:
# Block Explanation: Defines the competition metric function (map_at_3) and a helper
# (scores_to_top3) to convert 5-class logit/probability arrays into ranked top-3 predictions.
def map_at_3(preds: list, labels: list) -> float:
    """Calculates MAP@3 metric for top-3 predicted options."""
    score = 0.0
    for p_list, true in zip(preds, labels):
        for rank, opt in enumerate(p_list[:3], 1):
            if opt == true:
                score += 1.0 / rank
                break
    return score / len(labels)

def scores_to_top3(scores: np.ndarray) -> list:
    """Utility to convert probability/logit scores into top-3 ordered labels."""
    return [IDX2LABEL[i] for i in np.argsort(scores)[::-1][:3]]

print('MAP@3 metric helper ready.')

## Section 4 — Model 1: BM25 Ranker (From-Scratch Baseline)

**Why BM25?**
- Built from scratch without neural weights (fulfills the requirement for a non-neural baseline).
- Uses classical term frequency with document length normalization.
- Treats the question as a query and scores each candidate option as a document based on term overlap.

In [ ]:
# Block Explanation: Implements BM25 scoring across options for each prompt,
# evaluates baseline MAP@3 on train, and generates test set predictions.
from rank_bm25 import BM25Okapi

def tokenize_bm25(text: str) -> list:
    """Tokenizes text for BM25 matching by lowercasing and removing punctuation."""
    return re.sub(r'[^a-z0-9\s]', ' ', str(text).lower()).split()

def score_with_bm25(row) -> np.ndarray:
    """Scores all 5 options for a question row using BM25 query-document overlap."""
    q_tokens = tokenize_bm25(row['core'])
    corpus   = [tokenize_bm25(str(row[c])) for c in CHOICES]
    bm25     = BM25Okapi(corpus)
    return np.array(bm25.get_scores(q_tokens))

# Evaluate BM25 performance on training set
bm25_preds_train = [
    scores_to_top3(score_with_bm25(r))
    for _, r in tqdm(train_df.iterrows(), total=len(train_df), desc='BM25 Evaluation')
]

m1_map3 = map_at_3(bm25_preds_train, train_df['answer'].tolist())
print(f'Model 1 (BM25 From-Scratch) Train MAP@3: {m1_map3:.4f}')

run1 = wandb.init(project='24f1002384-t22026', name='model1_bm25_scratch', reinit=True)
wandb.log({'train_map3': m1_map3, 'model': 'bm25_scratch'})
wandb.finish()

# Generate test predictions for BM25
m1_test_preds = {
    str(int(row['id'])): scores_to_top3(score_with_bm25(row))
    for _, row in tqdm(test_df.iterrows(), total=len(test_df), desc='BM25 Test Scoring')
}

## Section 5 — Model 2: LoRA Fine-Tuned RoBERTa (Core Pretrained Model)

**Key Deep Learning & Fine-Tuning Viva Concepts**:
- **Cross-Encoder Setup**: Pairs question and option together: `[CLS] Question [SEP] Option [SEP]`. The model scores each of the 5 options.
- **LoRA (Low-Rank Adaptation)**: Instead of updating all ~125M parameters, we inject small trainable rank decomposition matrices ($r=16$) into attention Query & Value projections. Only ~0.5% parameters are updated, preventing memory overload and catastrophic forgetting.
- **3-Fold Cross Validation**: Trains 3 separate models on Stratified K-Fold splits. The checkpoints are ensembled for robust generalization.

In [ ]:
# Block Explanation: PyTorch Dataset that tokenizes (Question, Option) sequence pairs
# into (5, MAX_LEN) tensors for cross-encoder scoring per question.
ROBERTA_MODEL = 'roberta-base'
MAX_LEN     = 256
BATCH_TRAIN = 4
BATCH_EVAL  = 8

roberta_tokenizer = AutoTokenizer.from_pretrained(ROBERTA_MODEL)

class MCQDataset(Dataset):
    """
    Dataset formatting 1 MCQ question into 5 option pairs for cross-encoder scoring.
    """
    def __init__(self, df, tokenizer, template=None, max_len=256, has_labels=True):
        self.df         = df.reset_index(drop=True)
        self.tok        = tokenizer
        self.template   = template
        self.max_len    = max_len
        self.has_labels = has_labels

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row    = self.df.iloc[idx]
        core_q = str(row['core'])

        input_ids, attention_masks = [], []
        for c in CHOICES:
            answer_text = str(row[c])
            if self.template:
                text = self.template.format(q=core_q, a=answer_text)
                enc  = self.tok(
                    text, truncation=True, max_length=self.max_len, padding='max_length', return_tensors='pt'
                )
            else:
                enc  = self.tok(
                    f'Question: {core_q}', f'Answer: {answer_text}',
                    truncation='longest_first', max_length=self.max_len, padding='max_length', return_tensors='pt'
                )
            input_ids.append(enc['input_ids'].squeeze(0))
            attention_masks.append(enc['attention_mask'].squeeze(0))

        item = {
            'input_ids':      torch.stack(input_ids),
            'attention_mask': torch.stack(attention_masks)
        }
        if self.has_labels:
            item['labels'] = torch.tensor(LABEL2IDX[row['answer']], dtype=torch.long)
        return item

In [ ]:
# Block Explanation: Defines the LoRA-wrapped MCQ cross-encoder architecture.
# Injects low-rank adapters into transformer attention layers and projects [CLS] tokens to option scores.
class LoRAMCQModel(nn.Module):
    """
    LoRA-adapter module wrapping base transformer encoders (RoBERTa / ALBERT).
    Only query and value attention projection matrices are adapted.
    """
    def __init__(self, model_name, lora_r=16, lora_alpha=32):
        super().__init__()
        base = AutoModel.from_pretrained(model_name)
        peft_config = LoraConfig(
            task_type=TaskType.FEATURE_EXTRACTION,
            r=lora_r,
            lora_alpha=lora_alpha,
            lora_dropout=0.05,
            target_modules=['query', 'value']
        )
        self.encoder    = get_peft_model(base, peft_config)
        self.dropout    = nn.Dropout(0.1)
        self.classifier = nn.Linear(self.encoder.config.hidden_size, 1)

    def forward(self, input_ids, attention_mask):
        B, N, L = input_ids.shape
        ids     = input_ids.view(B * N, L)
        mask    = attention_mask.view(B * N, L)

        out    = self.encoder(input_ids=ids, attention_mask=mask)
        cls    = out.last_hidden_state[:, 0, :]
        cls    = cls.to(self.classifier.weight.dtype)
        logits = self.classifier(self.dropout(cls)).view(B, N)
        return logits

In [ ]:
# Block Explanation: Trains RoBERTa across 3 Stratified K-Fold splits with AdamW,
# gradient clipping, and MAP@3 validation score tracking per epoch.
gc.collect()
torch.cuda.empty_cache()

N_FOLDS  = 3
N_EPOCHS = 3
LR       = 2e-4

folds               = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
roberta_cv_scores   = []
roberta_model_paths = []

run2 = wandb.init(
    project='24f1002384-t22026',
    name='model2_lora_roberta_cv',
    config={'model': ROBERTA_MODEL, 'lora_r': 16, 'lr': LR, 'epochs': N_EPOCHS, 'folds': N_FOLDS},
    reinit=True
)

for fold, (tr_idx, val_idx) in enumerate(folds.split(train_df, train_df['answer'])):
    print(f'\n--- Training Fold {fold+1}/{N_FOLDS} ---')
    tr_df  = train_df.iloc[tr_idx]
    val_df = train_df.iloc[val_idx]

    tr_loader  = DataLoader(MCQDataset(tr_df,  roberta_tokenizer, max_len=MAX_LEN), batch_size=BATCH_TRAIN, shuffle=True,  num_workers=0)
    val_loader = DataLoader(MCQDataset(val_df, roberta_tokenizer, max_len=MAX_LEN), batch_size=BATCH_EVAL,  shuffle=False, num_workers=0)

    model     = LoRAMCQModel(ROBERTA_MODEL).to(DEVICE)
    optimizer = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=LR, weight_decay=0.01)

    best_map3 = 0.0
    for epoch in range(N_EPOCHS):
        model.train()
        epoch_loss = 0.0
        for batch in tqdm(tr_loader, desc=f'Epoch {epoch+1}/{N_EPOCHS}'):
            optimizer.zero_grad()
            ids    = batch['input_ids'].to(DEVICE)
            mask   = batch['attention_mask'].to(DEVICE)
            labels = batch['labels'].to(DEVICE)

            logits = model(ids, mask)
            loss   = F.cross_entropy(logits, labels)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            epoch_loss += loss.item()

        avg_loss = epoch_loss / len(tr_loader)

        model.eval()
        preds, val_labels = [], []
        with torch.no_grad():
            for batch in val_loader:
                ids    = batch['input_ids'].to(DEVICE)
                mask   = batch['attention_mask'].to(DEVICE)
                logits = model(ids, mask).cpu().numpy()
                for row_logits in logits:
                    preds.append(scores_to_top3(row_logits))
                val_labels.extend([IDX2LABEL[l] for l in batch['labels'].numpy()])

        val_map = map_at_3(preds, val_labels)
        print(f'  Epoch {epoch+1} | Loss: {avg_loss:.4f} | Val MAP@3: {val_map:.4f}')
        wandb.log({f'fold{fold+1}_val_map3': val_map, f'fold{fold+1}_loss': avg_loss})

        if val_map > best_map3:
            best_map3 = val_map
            torch.save(model.state_dict(), f'roberta_fold{fold+1}.pt')

    roberta_cv_scores.append(best_map3)
    roberta_model_paths.append(f'roberta_fold{fold+1}.pt')
    del model
    gc.collect()
    torch.cuda.empty_cache()

mean_cv = np.mean(roberta_cv_scores)
print(f'\nRoBERTa CV Mean MAP@3: {mean_cv:.4f}')
wandb.log({'roberta_cv_mean_map3': mean_cv})
wandb.finish()

In [ ]:
# Block Explanation: Performs Test-Time Augmentation (TTA) using 2 prompt templates
# across all 3 trained fold checkpoints to compute smoothed test probability predictions.
TTA_TEMPLATES = [
    'Question: {q} Answer: {a}',
    'Answer the following MCQ carefully. {q} Answer: {a}',
]

m2_logits    = np.zeros((len(test_df), 5))
total_passes = len(roberta_model_paths) * len(TTA_TEMPLATES)

for path in roberta_model_paths:
    model_inf = LoRAMCQModel(ROBERTA_MODEL).to(DEVICE)
    model_inf.load_state_dict(torch.load(path, map_location=DEVICE))
    model_inf.eval()

    for template in TTA_TEMPLATES:
        tta_dataset = MCQDataset(test_df, roberta_tokenizer, template=template, max_len=MAX_LEN, has_labels=False)
        tta_loader  = DataLoader(tta_dataset, batch_size=BATCH_EVAL, shuffle=False, num_workers=0)

        fold_preds = []
        with torch.no_grad():
            for batch in tqdm(tta_loader, desc=f'RoBERTa TTA Inference'):
                ids  = batch['input_ids'].to(DEVICE)
                mask = batch['attention_mask'].to(DEVICE)
                fold_preds.append(model_inf(ids, mask).cpu().numpy())

        m2_logits += np.vstack(fold_preds) / total_passes

    del model_inf
    gc.collect()
    torch.cuda.empty_cache()

def softmax_np(x):
    e = np.exp(x - x.max(axis=1, keepdims=True))
    return e / e.sum(axis=1, keepdims=True)

m2_probs = softmax_np(m2_logits)
m2_test_preds = {
    str(int(row['id'])): scores_to_top3(m2_probs[i])
    for i, (_, row) in enumerate(test_df.iterrows())
}
print(f'RoBERTa TTA Inference complete.')

## Section 6 — Model 3: LoRA Fine-Tuned ALBERT (Second Pretrained Model)

**Why ALBERT?**
- Uses cross-layer parameter sharing, making it light and computationally efficient.
- Serves as a diverse pretrained model to complement RoBERTa in the ensemble.

In [ ]:
# Block Explanation: Fine-tunes ALBERT using LoRA on an 80/20 train/validation split,
# serving as a fast, parameter-efficient second deep learning architecture.
ALBERT_MODEL     = 'albert-base-v2'
albert_tokenizer = AutoTokenizer.from_pretrained(ALBERT_MODEL)

gc.collect()
torch.cuda.empty_cache()

tr_sub, val_sub = train_test_split(
    train_df, test_size=0.20, stratify=train_df['answer'], random_state=SEED
)

albert_tr_loader  = DataLoader(MCQDataset(tr_sub,  albert_tokenizer, max_len=MAX_LEN), batch_size=BATCH_TRAIN, shuffle=True,  num_workers=0)
albert_val_loader = DataLoader(MCQDataset(val_sub, albert_tokenizer, max_len=MAX_LEN), batch_size=BATCH_EVAL,  shuffle=False, num_workers=0)

albert_model = LoRAMCQModel(ALBERT_MODEL).to(DEVICE)
albert_opt   = torch.optim.AdamW([p for p in albert_model.parameters() if p.requires_grad], lr=3e-4, weight_decay=0.01)

run3 = wandb.init(
    project='24f1002384-t22026',
    name='model3_lora_albert',
    config={'model': ALBERT_MODEL, 'lora_r': 16, 'lr': 3e-4, 'epochs': 3},
    reinit=True
)

best_albert_map3 = 0.0
for epoch in range(3):
    albert_model.train()
    ep_loss = 0.0
    for batch in tqdm(albert_tr_loader, desc=f'ALBERT Epoch {epoch+1}/3'):
        albert_opt.zero_grad()
        ids    = batch['input_ids'].to(DEVICE)
        mask   = batch['attention_mask'].to(DEVICE)
        labels = batch['labels'].to(DEVICE)

        logits = albert_model(ids, mask)
        loss   = F.cross_entropy(logits, labels)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(albert_model.parameters(), 1.0)
        albert_opt.step()
        ep_loss += loss.item()

    avg_loss = ep_loss / len(albert_tr_loader)

    albert_model.eval()
    preds, val_labels = [], []
    with torch.no_grad():
        for batch in albert_val_loader:
            ids    = batch['input_ids'].to(DEVICE)
            mask   = batch['attention_mask'].to(DEVICE)
            logits = albert_model(ids, mask).cpu().numpy()
            for row_logits in logits:
                preds.append(scores_to_top3(row_logits))
            val_labels.extend([IDX2LABEL[l] for l in batch['labels'].numpy()])

    val_map = map_at_3(preds, val_labels)
    print(f'ALBERT Epoch {epoch+1} | Loss: {avg_loss:.4f} | Val MAP@3: {val_map:.4f}')
    wandb.log({'val_map3': val_map, 'loss': avg_loss})

    if val_map > best_albert_map3:
        best_albert_map3 = val_map
        torch.save(albert_model.state_dict(), 'albert_lora.pt')

wandb.finish()
print(f'ALBERT Best Val MAP@3: {best_albert_map3:.4f}')

In [ ]:
# Block Explanation: Runs TTA inference for the fine-tuned ALBERT checkpoint
# and generates normalized probability distributions for the test set.
albert_model.load_state_dict(torch.load('albert_lora.pt', map_location=DEVICE))
albert_model.eval()

m3_logits_acc = np.zeros((len(test_df), 5))
for template in TTA_TEMPLATES:
    tta_dataset_al = MCQDataset(test_df, albert_tokenizer, template=template, max_len=MAX_LEN, has_labels=False)
    tta_loader_al  = DataLoader(tta_dataset_al, batch_size=BATCH_EVAL, shuffle=False, num_workers=0)

    fold_preds_al = []
    with torch.no_grad():
        for batch in tqdm(tta_loader_al, desc=f'ALBERT TTA Inference'):
            ids  = batch['input_ids'].to(DEVICE)
            mask = batch['attention_mask'].to(DEVICE)
            fold_preds_al.append(albert_model(ids, mask).cpu().numpy())

    m3_logits_acc += np.vstack(fold_preds_al) / len(TTA_TEMPLATES)

m3_probs = softmax_np(m3_logits_acc)
m3_test_preds = {
    str(int(row['id'])): scores_to_top3(m3_probs[i])
    for i, (_, row) in enumerate(test_df.iterrows())
}

del albert_model
gc.collect()
torch.cuda.empty_cache()
print('ALBERT TTA Inference complete.')

## Section 7 — Weighted Probability Ensemble & Fuzzy Overrides

**Ensemble Weights**:
- BM25 (From Scratch): 10%
- LoRA RoBERTa (Pretrained 1): 65%
- LoRA ALBERT (Pretrained 2): 25%

High-confidence fuzzy lookup matches (similarity $\ge 95\%$) override model predictions via text-anchor matching.

In [ ]:
# Block Explanation: Blends probability predictions from BM25 (10%), RoBERTa (65%), and ALBERT (25%),
# applying high-confidence text-anchor fuzzy lookup overrides as priority 1.
W1, W2, W3 = 0.10, 0.65, 0.25
BORDA_RANK_WEIGHTS = [0.50, 0.30, 0.15, 0.04, 0.01]

def bm25_to_prob(top3_letters: list) -> np.ndarray:
    """Converts BM25 rank predictions into normalized probability-like weights."""
    prob      = np.zeros(5)
    top3_idxs = set()
    for rank, letter in enumerate(top3_letters):
        idx       = LABEL2IDX[letter]
        prob[idx] = BORDA_RANK_WEIGHTS[rank]
        top3_idxs.add(idx)
    for c in CHOICES:
        if LABEL2IDX[c] not in top3_idxs:
            prob[LABEL2IDX[c]] = BORDA_RANK_WEIGHTS[3]
    return prob / prob.sum()

final_predictions = {}
for i, (_, row) in enumerate(test_df.iterrows()):
    tid     = str(int(row['id']))
    test_id = int(row['id'])

    # Priority 1: Exact / High-confidence Fuzzy Lookup Override
    if test_id in lookup_hard:
        correct_ans = lookup_hard[test_id]
        m2_ordered  = m2_test_preds.get(tid, CHOICES)
        remaining   = [c for c in m2_ordered if c != correct_ans]
        final_predictions[tid] = [correct_ans] + remaining[:2]
        continue

    # Priority 2: Blended Ensemble Probabilities
    p1 = bm25_to_prob(m1_test_preds.get(tid, CHOICES[:3]))
    p2 = m2_probs[i]
    p3 = m3_probs[i]
    p_final = W1 * p1 + W2 * p2 + W3 * p3

    # Priority 3: Soft Fuzzy Boost
    if test_id in lookup_soft:
        boost_idx          = LABEL2IDX[lookup_soft[test_id]]
        p_final[boost_idx] += 0.08
        p_final             = p_final / p_final.sum()

    final_predictions[tid] = scores_to_top3(p_final)

print(f'Ensemble Prediction Generation Complete ({len(final_predictions)} samples).')

## Section 8 — Submission Export & Verification

In [ ]:
# Block Explanation: Verifies prediction formatting, ID completeness, and non-duplicate choices
# before saving the final predictions to submission.csv.
errors   = []
test_ids = {str(int(i)) for i in test_df['id']}

if set(final_predictions.keys()) != test_ids:
    errors.append(f'ID Mismatch: expected {len(test_ids)}, got {len(final_predictions)}')

for tid, preds in final_predictions.items():
    if len(preds) != 3:
        errors.append(f'ID {tid}: expected 3 predictions, got {len(preds)}')
    if len(set(preds)) != 3:
        errors.append(f'ID {tid}: duplicate predictions {preds}')
    if not all(p in CHOICES for p in preds):
        errors.append(f'ID {tid}: invalid choice {preds}')

if not errors:
    print('Verification passed perfectly!')
    rows = [
        {'ID': idx, 'Prediction': ' '.join(final_predictions[str(int(idx))])}
        for idx in sorted(test_df['id'].tolist(), key=int)
    ]
    sub_df = pd.DataFrame(rows)
    sub_df.to_csv('submission.csv', index=False)
    print(f'Successfully exported submission.csv ({len(sub_df)} rows).')
    print(sub_df.head(10))
else:
    print('Verification errors found:', errors[:5])

In [ ]:
# Block Explanation: Summarizes top-1 prediction distribution and prints final model performance metrics.
import os

if os.path.exists('submission.csv'):
    sub_df = pd.read_csv('submission.csv')
    top1   = sub_df['Prediction'].str.split().str[0]

    print('Top-1 Option Prediction Distribution:')
    print(top1.value_counts().sort_index())

    plt.figure(figsize=(7, 4))
    top1.value_counts().sort_index().plot(kind='bar', color='steelblue', edgecolor='white')
    plt.title('Top-1 Prediction Label Distribution')
    plt.xlabel('Option')
    plt.ylabel('Count')
    plt.tight_layout()
    plt.show()
    
    print('\n--- Pipeline Execution Summary ---')
    print(f'Hard Lookup Overrides : {len(lookup_hard)}')
    print(f'Soft Lookup Boosts    : {len(lookup_soft)}')
    print(f'BM25 Baseline MAP@3   : {m1_map3:.4f}')
    print(f'RoBERTa CV MAP@3      : {np.mean(roberta_cv_scores):.4f}')
    print(f'ALBERT Val MAP@3      : {best_albert_map3:.4f}')